<a href="https://colab.research.google.com/github/gulshan-4/Mobile-Store-project/blob/master/notebooks/piper_multilingual_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="ffc800"> **[Piper](https://github.com/rhasspy/piper) training notebook.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)

---

- Notebook made by [rmcpantoja](http://github.com/rmcpantoja)
- Collaborator: [Xx_Nessu_xX](http://github.com/Xx_Nessu_xX)

---

# Notes:

- <font color="orange">**Things in orange mean that they are important.**

# Credits:

* [Feanix-Fyre fork](https://github.com/Feanix-Fyre/piper) with some improvements.
* [Tacotron2 NVIDIA training notebook](https://github.com/justinjohn0306/FakeYou-Tacotron2-Notebook) - Dataset duration snippet.
* [🐸TTS](https://github.com/coqui-ai/TTS) - Resampler and XTTS formater demo.

# <font color="ffc800">🔧 ***First steps.*** 🔧

In [1]:
#@markdown ## <font color="ffc800"> **Google Colab Anti-Disconnect.** 🔌
#@markdown ---
#@markdown #### Avoid automatic disconnection. Still, it will disconnect after <font color="orange">**6 to 12 hours**</font>.

import IPython
js_code = '''
function ClickConnect(){
console.log("Working");
document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect,60000)
'''
display(IPython.display.Javascript(js_code))

<IPython.core.display.Javascript object>

In [2]:
#@markdown ## <font color="ffc800"> **Check GPU type.** 👁️
#@markdown ---
#@markdown #### A higher capable GPU can lead to faster training speeds. By default, you will have a <font color="orange">**Tesla T4**</font>.
!nvidia-smi

Wed Aug 26 14:36:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
#@markdown # <font color="ffc800"> **Mount Google Drive.** 📂
#@markdown ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
#@markdown # <font color="ffc800"> **Install software.** 📦
#@markdown ---
#@markdown ####In this cell the synthesizer and its necessary dependencies to execute the training will be installed. (this may take a while)

import os

print("🧹 Cleaning up old files...")
if os.path.exists("/content/piper"):
    !rm -rf /content/piper

print("📥 Cloning Piper repository...")
# Removed -q (quiet) so you can see the download progress
!git clone https://github.com/rmcpantoja/piper
%cd /content/piper/src/python

print("📥 Downloading resample.py...")
!wget "https://raw.githubusercontent.com/coqui-ai/TTS/dev/TTS/bin/resample.py"

print("📦 Installing Python dependencies (this will print a lot of text)...")
# Removed -q so pip shows you what it's doing
!pip install cython>=0.29.0 piper-phonemize-fix librosa>=0.9.2 numpy==1.26.4 onnxruntime>=1.15.0 pytorch-lightning
!pip install --upgrade gdown transformers

print("⚙️ Compiling Monotonic Align (WARNING: This takes 1-3 minutes. Just let it spin!)...")
!bash build_monotonic_align.sh

# Useful vars:
use_whisper = True
print("✅ Done!")

🧹 Cleaning up old files...
📥 Cloning Piper repository...
Cloning into 'piper'...
remote: Enumerating objects: 3004, done.
remote: Counting objects: 100% (457/457), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 3004 (delta 418), reused 411 (delta 411), pack-reused 2547 (from 1)
Receiving objects: 100% (3004/3004), 217.95 MiB | 18.32 MiB/s, done.
Resolving deltas: 100% (1795/1795), done.
/content/piper/src/python
📥 Downloading resample.py...
--2026-08-26 14:40:06--  https://raw.githubusercontent.com/coqui-ai/TTS/dev/TTS/bin/resample.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2776 (2.7K) [text/plain]
Saving to: ‘resample.py’

resample.py         100%[===================>]   2.71K  --.-KB/s    in 0s      

2026-08-26 14:40:07 (42.5

In [6]:
!pip install piper-phonemize-fix

  Using cached piper_phonemize_fix-1.2.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (367 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 63.0 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


# <font color="ffc800"> 🤖 ***Training.*** 🤖

In [7]:
#@markdown # <font color="ffc800"> **1. Extract dataset.** 📥
#@markdown ---
#@markdown ####Important: the audios must be in <font color="orange">**wav format, (16000 or 22050hz, 16-bits, mono), and, for convenience, numbered. Example:**

#@markdown * <font color="orange">**1.wav**</font>
#@markdown * <font color="orange">**2.wav**</font>
#@markdown * <font color="orange">**3.wav**</font>
#@markdown * <font color="orange">**.....**</font>

#@markdown ---
import os
import wave
import zipfile
import datetime

def get_dataset_duration(wav_path):
    totalduration = 0
    for file_name in [x for x in os.listdir(wav_path) if os.path.isfile(x) and ".wav" in x]:
        with wave.open(file_name, "rb") as wave_file:
            frames = wave_file.getnframes()
            rate = wave_file.getframerate()
            duration = frames / float(rate)
            totalduration += duration
    wav_count = len(os.listdir(wav_path))
    duration_str = str(datetime.timedelta(seconds=round(totalduration, 0)))
    return wav_count, duration_str

%cd /content
if not os.path.exists("/content/dataset"):
    os.makedirs("/content/dataset")
    os.makedirs("/content/dataset/wavs")
%cd /content/dataset
#@markdown ### Audio dataset path to unzip:
zip_path = "/content/drive/MyDrive/PiperTraining/lm-audio.zip" #@param {type:"string"}
zip_path = zip_path.strip()
if zip_path:
    if os.path.exists(zip_path):
        if zipfile.is_zipfile(zip_path):
            print("Unzipping audio content...")
            !unzip -q -j "{zip_path}" -d /content/dataset/wavs
        else:
            print("Copying audio contents of this folder...")
            fp = zip_path + "/."
            !cp -a "$fp" "/content/dataset/wavs"
    else:
        raise Exception("The path provided to the wavs is not correct. Please set a valid path.")
else:
    raise Exception("You must provide with a path to the wavs.")
if os.path.exists("/content/dataset/wavs/wavs"):
    for file in os.listdir("/content/dataset/wavs/wavs"):
        !mv /content/dataset/wavs/wavs/"$file"  /content/dataset/wavs/"$file"
    !rm -r /content/dataset/wavs/*.txt
    !rm -r /content/dataset/wavs/*.csv
%cd /content/dataset/wavs
audio_count, dataset_dur = get_dataset_duration("/content/dataset/wavs")
print(f"Opened dataset with {audio_count} wavs with duration {dataset_dur}.")
%cd ..
#@markdown ---

/content
/content/dataset
Unzipping audio content...
/content/dataset/wavs
Opened dataset with 85 wavs with duration 0:09:35.
/content/dataset


In [8]:
#@markdown # <font color="ffc800"> **2. Upload the transcript file.** 📝
#@markdown ---
#@markdown ####<font color="orange">**Important: the transcription means writing what the character says in each of the audios, and it must have the following structure:**

#@markdown ##### <font color="orange">For a single-speaker dataset:
#@markdown * wavs/1.wav|This is what my character says in audio 1.
#@markdown * wavs/2.wav|This, the text that the character says in audio 2.
#@markdown * ...

#@markdown ##### <font color="orange">For a multi-speaker dataset:

#@markdown * wavs/speaker1audio1.wav|speaker1|This is what the first speaker says.
#@markdown * wavs/speaker1audio2.wav|speaker1|This is another audio of the first speaker.
#@markdown * wavs/speaker2audio1.wav|speaker2|This is what the second speaker says in the first audio.
#@markdown * wavs/speaker2audio2.wav|speaker2|This is another audio of the second speaker.
#@markdown * ...

#@markdown And so on. In addition, the transcript must be in a <font color="orange">**.csv or .txt format. (UTF-8 without BOM)**

#@markdown ## Auto-transcribe with whisper if transcription is not provided.

#@markdown **Note: If you don't upload any transcription files, the wavs will be transcribed using the whisper tool when you execute the next step. Then, the notebook will continue with the rest of the preprocessing if there are no errors. Although the Whisper tool has good transcription results, in my experience I recommend transcribing manually and uploading it from this cell, since a good TTS voice needs to be optimized to give even better results. For example, when transcribing manually you will be able to observe every detail that the speaker makes (such as punctuation, sounds, etc.), and capture them in the transcription according to the speaker's intonations.**


#@markdown However, if you want to transcribe and review this transcription, you can use the individual notebooks:

#@markdown * [English](http://colab.research.google.com/github/rmcpantoja/My-Colab-Notebooks/blob/main/notebooks/OpenAI_Whisper_-_DotCSV_(Speech_dataset_multi-transcryption_support)en.ipynb)
#@markdown * [French](http://colab.research.google.com/github/rmcpantoja/My-Colab-Notebooks/blob/main/notebooks/OpenAI_Whisper_-_DotCSV_(Speech_dataset_multi-transcryption_support)fr.ipynb)
#@markdown * [Spanish](http://colab.research.google.com/github/rmcpantoja/My-Colab-Notebooks/blob/main/notebooks/OpenAI_Whisper_-_DotCSV_(Speech_dataset_multi-transcryption_support)es.ipynb)

#@markdown ---
%cd /content/dataset
from google.colab import files
!rm /content/dataset/metadata.csv

if os.path.exists("/content/dataset/wavs/_transcription.txt"):
  !mv "/content/dataset/wavs/_transcription.txt" metadata.csv
else:
  listfn, length = files.upload().popitem()
  if listfn != "metadata.csv":
    !mv "$listfn" metadata.csv

use_whisper = False
%cd ..

/content/dataset
rm: cannot remove '/content/dataset/metadata.csv': No such file or directory


Saving metadata.csv to metadata.csv
/content


In [9]:
#@markdown # <font color="ffc800"> **3. Preprocess dataset.** 🔄
#@markdown ---
import os

# --- NEW SAFETY CHECK ---
print("📦 Verifying required packages...")
!pip install -q onnxruntime faster-whisper

if not os.path.exists("/content/piper/src/python"):
    raise Exception("❌ ERROR: Piper folder is missing! You need to go back and run the 'Install software' cell to completion first.")
# ------------------------

# Fallback just in case you restarted your Colab runtime
if "use_whisper" not in locals():
    use_whisper = True

if use_whisper:
    import torch
    from faster_whisper import WhisperModel
    from tqdm import tqdm
    from google import colab

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    def make_dataset(path, language):
        metadata = ""
        text = ""
        files = [f for f in os.listdir(path) if f.endswith(".wav")]
        assert len(files) > 0, "You don't have wavs uploaded either! Please upload at least one zip with the wavs in step 2."
        metadata_file = open(f"{path}/../metadata.csv", "w")
        whisper = WhisperModel("large-v3", device=device, compute_type="float16")
        for audio_file in tqdm(files):
            full_path = os.path.join(path, audio_file)
            segments, _ = whisper.transcribe(full_path, word_timestamps=False, language=language)
            for segment in segments:
                text += segment.text
            text = text.strip()
            text = text.replace('\n', ' ')
            metadata = f"{audio_file}|{text}\n"
            metadata_file.write(metadata)
            text = ""
        colab.files.download(f"{path}/../metadata.csv")
        del whisper
        return True

#@markdown ### First of all, select the language of your dataset.
language = "Hindi" #@param ["Hindi", "Català", "čeština", "Dansk", "Deutsch", "Ελληνικά", "English (British)", "English (U.S.)", "Español (Castellano)", "Español (Latinoamericano)", "Suomi", "Français", "Magyar", "Icelandic", "Italiano", "ქართული", "қазақша", "Lëtzebuergesch", "नेपाली", "Nederlands", "Norsk", "Polski", "Português (Brasil)", "Português (Portugal)", "Română", "Русский", "Српски", "Svenska", "Kiswahili", "Türkçe", "украї́нська", "Tiếng Việt", "简体中文"]
#@markdown ---
# language definition:
languages = {
    "Hindi": "hi",
    "Català": "ca",
    "čeština": "cs",
    "Dansk": "da",
    "Deutsch": "de",
    "Ελληνικά": "el",
    "English (British)": "en",
    "English (U.S.)": "en-us",
    "Español (Castellano)": "es",
    "Español (Latinoamericano)": "es-419",
    "Suomi.": "fi",
    "Français": "fr",
    "Magyar": "hu",
    "Icelandic": "is",
    "Italiano": "it",
    "ქართული": "ka",
    "қазақша": "kk",
    "Lëtzebuergesch": "lb",
    "नेपाली": "ne",
    "Nederlands": "nl",
    "Norsk": "nb",
    "Polski": "pl",
    "Português (Brasil)": "pt-br",
    "Português (Portugal)": "pt-pt",
    "Română": "ro",
    "Русский": "ru",
    "Српски": "sr",
    "Svenska": "sv",
    "Kiswahili": "sw",
    "Türkçe": "tr",
    "украї́нська": "uk",
    "Tiếng Việt": "vi",
    "简体中文": "zh"
}

def _get_language(code):
    return languages[code]

final_language = _get_language(language)
#@markdown ### Choose a name for your model:
model_name = "kiran-med2" #@param {type:"string"}
#@markdown ---
# output:
#@markdown ### Choose the working folder: (recommended to save to Drive)

#@markdown The working folder will be used in preprocessing, but also in training the model.
output_path = "/content/drive/MyDrive/colab/piper" #@param {type:"string"}
output_dir = output_path+"/"+model_name
if not os.path.exists(output_dir):
  os.makedirs(output_dir)
#@markdown ---
#@markdown ### Choose dataset format:
dataset_format = "ljspeech" #@param ["ljspeech", "mycroft"]
#@markdown ---
#@markdown ### Is this a single speaker dataset? Otherwise, uncheck:
single_speaker = True #@param {type:"boolean"}
if single_speaker:
  force_sp = " --single-speaker"
else:
  force_sp = ""
#@markdown ---
#@markdown ### Select the sample rate of the dataset:
sample_rate = "22050" #@param ["16000", "22050"]
#@markdown ---
# creating paths:
if not os.path.exists("/content/audio_cache"):
    os.makedirs("/content/audio_cache")
%cd /content/piper/src/python
#@markdown ### Do you want to train using this sample rate, but your audios don't have it?
#@markdown The resampler helps you do it quickly!
resample = False #@param {type:"boolean"}
if resample:
  !python resample.py --input_dir "/content/dataset/wavs" --output_dir "/content/dataset/wavs_resampled" --output_sr {sample_rate} --file_ext "wav"
  !mv /content/dataset/wavs_resampled/* /content/dataset/wavs
#@markdown ---
# check transcription:
if use_whisper:
    print("Transcript file hasn't been uploaded. Transcribing these audios using Whisper...")
    make_dataset("/content/dataset/wavs", final_language[:2])
    print("Transcription done! Pre-processing...")
!python -m piper_train.preprocess \
  --language {final_language} \
  --input-dir /content/dataset \
  --cache-dir "/content/audio_cache" \
  --output-dir "{output_dir}" \
  --dataset-name "{model_name}" \
  --dataset-format {dataset_format} \
  --sample-rate {sample_rate} \
  {force_sp}
print("Preprocessing done!")

📦 Verifying required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 18.1 MB/s eta 0:00:00
/content/piper/src/python
INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 83 utterance(s) with 2 worker(s)
Preprocessing done!


In [11]:
#@markdown # <font color="ffc800"> **4. Settings.** 🧰
#@markdown ---
import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import output
import os
import re
import glob

#@markdown ### <font color="orange">**Select the action to train this dataset: (READ CAREFULLY)**
action = "finetune" #@param ["Continue training", "convert single-speaker to multi-speaker model", "finetune", "train from scratch"]
#@markdown ---

# Fallback definition just in case runtime restarted
try:
    final_language
except NameError:
    final_language = "hi"

if action == "Continue training":
    checkpoints = glob.glob(f"{output_dir}/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
    if len(checkpoints):
        last_checkpoint = sorted(checkpoints, key=lambda x: int(re.findall(r'version_(\d+)', x)[0]))[-1]
        ft_command = f'--resume_from_checkpoint "{last_checkpoint}" '
        print(f"Continuing {model_name}'s training at: {last_checkpoint}")
    else:
        raise Exception("Training cannot be continued as there is no checkpoint to continue at.")
elif action == "finetune":
    if os.path.exists(f"{output_dir}/lightning_logs/version_0/checkpoints/last.ckpt"):
        raise Exception("Oh no! You have already trained this model before, you cannot choose this option since your progress will be lost. Please select 'Continue training'.")
    else:
        ft_command = '--resume_from_checkpoint "/content/pretrained.ckpt" '
elif action == "convert single-speaker to multi-speaker model":
    if not single_speaker:
        ft_command = '--resume_from_single_speaker_checkpoint "/content/pretrained.ckpt" '
    else:
        raise Exception("This dataset is not a multi-speaker dataset!")
else:
    ft_command = ""

if action == "convert single-speaker to multi-speaker model" or action == "finetune":
    # ---------------------------------------------------------
    # NEW: Custom fallback targeting the "Rohan" Hindi model
    # ---------------------------------------------------------
    if final_language == "hi":
        print("\033[93mNo Hindi model found in database. Automatically downloading official Piper Hindi (Rohan) checkpoint from Hugging Face...")
        # Direct link to the Rohan training checkpoint in Piper's dataset repo
        !wget -q "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/hi/hi_IN/rohan/medium/epoch%3D3190-step%3D309852.ckpt" -O "/content/pretrained.ckpt"

        if os.path.exists("/content/pretrained.ckpt"):
            print("\033[93mRohan base model downloaded successfully!")
        else:
            raise Exception("Couldn't download the fallback pretrained model!")
    else:
        try:
            with open('/content/piper/notebooks/pretrained_models.json') as f:
                pretrained_models = json.load(f)

            if final_language in pretrained_models:
                models = pretrained_models[final_language]
                model_options = [(name, name) for name, url in models.items()]
                model_dropdown = widgets.Dropdown(description="Choose pretrained model", options=model_options)
                download_button = widgets.Button(description="Download")

                def download_model(btn):
                    model_name = model_dropdown.value
                    model_url = pretrained_models[final_language][model_name]
                    print("\033[93mDownloading pretrained model...")
                    if model_url.startswith("1") or model_url.startswith("https://drive.google.com/file/d/"):
                        !gdown -q "{model_url}" -O "/content/pretrained.ckpt" --fuzzy
                    else:
                        !wget -q "{model_url}" -O "/content/pretrained.ckpt"

                    model_dropdown.close()
                    download_button.close()
                    output.clear()

                    if os.path.exists("/content/pretrained.ckpt"):
                        print("\033[93mModel downloaded!")
                    else:
                        raise Exception("Couldn't download the pretrained model!")

                download_button.on_click(download_model)
                display(model_dropdown, download_button)
            else:
                raise Exception(f"There are no pretrained models available for the language {final_language}. Try changing the action to 'train from scratch'.")
        except FileNotFoundError:
            raise Exception("The pretrained_models.json file was not found.")
else:
    print("\033[93mWarning: this model will be trained from scratch. You need at least 8 hours of data for everything to work decent. Good luck!")

#@markdown ### Choose batch size based on this dataset:
batch_size = 12 #@param {type:"integer"}
#@markdown ---
#@markdown ### Choose the quality for this model:
quality = "medium" #@param ["high", "x-low", "medium"]
#@markdown ---
#@markdown ### For how many epochs to save training checkpoints?
checkpoint_epochs = 480 #@param {type:"integer"}
#@markdown ---
#@markdown ### Interval to save best k models:
num_ckpt = 1 #@param {type:"integer"}
#@markdown ---
#@markdown ### Save latest model:
save_last = False # @param {type:"boolean"}
#@markdown ---
#@markdown ### Step interval to generate model samples:
log_every_n_steps = 1000 #@param {type:"integer"}
#@markdown ---
#@markdown ### Training epochs:
max_epochs = 10000 #@param {type:"integer"}

No Hindi model found in database. Automatically downloading official Piper Hindi (Rohan) checkpoint from Hugging Face...
Rohan base model downloaded successfully!


In [ ]:
#@markdown # <font color="ffc800"> **5. Run the TensorBoard extension.** 📈
#@markdown ---
#@markdown The TensorBoard is used to visualize the results of the model while it's being trained such as audio and losses.

%load_ext tensorboard
%tensorboard --logdir {output_dir}

In [ ]:
#@markdown # <font color="ffc800"> **6. Train.** 🏋️‍♂️
#@markdown ---
#@markdown ### Run this cell to train your final model!
import os
import glob
import sys

# ---------------------------------------------------------
# NEW: SAFETY CHECKS & PYTORCH 2.6 PATCHES FOR TRAINING
# ---------------------------------------------------------
print("📦 Verifying PyTorch Lightning and missing dependencies...")
!pip install -q pytorch-lightning tensorboard setuptools

print("🛠️ Applying PyTorch 2.6+ environment patches for Training...")
# 1. Patch pkgutil for Python 3.13 compatibility
import pkgutil
pkgutil_path = getattr(pkgutil, '__file__', None)
if pkgutil_path and os.path.exists(pkgutil_path):
    with open(pkgutil_path, 'r') as f:
        content = f.read()
    if 'class ImpImporter' not in content:
        with open(pkgutil_path, 'a') as f:
            f.write('\n# Added to fix Python 3.12+ compatibility\nclass ImpImporter:\n    pass\n')

# 2. Patch PyTorch Lightning to bypass PyTorch 2.6+ strict pickling checks
pl_paths = glob.glob("/usr/local/lib/python*/dist-packages/lightning_fabric/utilities/cloud_io.py") + \
           glob.glob("/usr/local/lib/python*/dist-packages/pytorch_lightning/utilities/cloud_io.py")

for path in pl_paths:
    if os.path.exists(path):
        with open(path, "r") as f:
            content = f.read()
        if "weights_only=False" not in content:
            content = content.replace(
                "torch.load(f, map_location=map_location)",
                "torch.load(f, map_location=map_location, weights_only=False)"
            )
            with open(path, "w") as f:
                f.write(content)
# ---------------------------------------------------------

# Fallback in case cell state was lost
try: save_last
except NameError: save_last = False

#@markdown ---
#@markdown ### <font color="orange">**Disable validation?**
#@markdown By uncheck this checkbox, this will allow to train the full dataset, without using any audio files or examples as a validation set. So, it will not be able to generate audios on the tensorboard while it's training. It is recommended to disable validation on extremely small datasets.
validation = True #@param {type:"boolean"}
if validation:
    validation_split = 0.05
    num_test_examples = 1
else:
    validation_split = 0
    num_test_examples = 0

if not save_last:
    save_last_command = ""
else:
    save_last_command = "--save_last True "

get_ipython().system(f'''
python -m piper_train \
--dataset-dir "{output_dir}" \
--accelerator 'gpu' \
--devices 1 \
--batch-size {batch_size} \
--validation-split {validation_split} \
--num-test-examples {num_test_examples} \
--quality {quality} \
--checkpoint-epochs {checkpoint_epochs} \
--num_ckpt {num_ckpt} \
{save_last_command}\
--log_every_n_steps {log_every_n_steps} \
--max_epochs {max_epochs} \
{ft_command}\
--precision 32
''')

📦 Verifying PyTorch Lightning and missing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
🛠️ Applying PyTorch 2.6+ environment patches for Training...
DEBUG:piper_train:Namespace(dataset_dir='/content/drive/MyDrive/colab/piper/kiran-med2', checkpoint_epochs=480, quality='medium', resume_from_single_speaker_checkpoint=None, batch_size=12, validation_split=0.05, num_test_examples=1, max_phoneme_ids=None, hidden_channels=192, inter_channels=192, filter_channels=768, n_layers=6, n_heads=2, lr_decay=0.999875, lr_reduce_enabled=False, lr_reduce_factor=0.5, l

#  <font color="orange">**Have you finished training and want to test the model?**

* If you want to run this model in any software that Piper integrates or the same Piper app, export your model using the [model exporter notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_model_exporter.ipynb)!
* Wait! I want to test this right now before exporting it to the supported format for Piper. Test your generated last.ckpt with [this notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_inference_(ckpt).ipynb)!